# Porting Morphological Deconvolution from Wang et al.

The major function is `expressionInAnnotations`. 

In [ ]:
import numpy as np
import anndata as ad
import polars as pl
from anndata import AnnData

from scipy.optimize import minimize
from scipy.special import gammaln

In [77]:
def negbinom_loglik(x, mu, size) :
    '''
    Negative binomial log-likelihood
    '''
    # display(f'x: {x.shape} \n mu: {mu.shape} \n size: {size.shape}')

    return (
        gammaln(x + size)
        - gammaln(size)
        - gammaln(x + 1)
        + size * np.log(size / (size + mu))
        + x * np.log(mu / (size + mu))
    )

In [78]:
def fit_gene_weights(x_col, f, sc, sigma, init):
    '''
    Optimize weights for a single gene (col of X) 
    '''

    def objective(p_log, x_col, f, sc):
        p = np.exp(p_log)
        sigma = p[0] + 0.2
        w = p[1:]
        mu = sc[:, None] * (f @ w)

        # print('mu pre-flat', mu.shape)
        return -np.sum(
            # negbinom_loglik(x_col, mu.flatten(), np.minimum(sigma, 1e3))
            negbinom_loglik(x_col, mu, np.minimum(sigma, 1e3))
        )
    
    init_log = np.log(np.concatenate([[max(sigma - 0.2, 1)], init]))
    res = minimize(objective, init_log, method='BFGS', args=(x_col, f, sc))
    p_opt = np.exp(res.x)

    return res.fun, p_opt[0] + 0.2, p_opt[1:]

In [79]:
def expression_in_annotations(x, f, sigma=None, max_iter=5, verbose=True):
    n_spots, n_genes = x.shape
    n_factors = f.shape[1]

    if sigma is None:
        sigma = np.ones(n_genes)
    sigma = np.clip(sigma, 0.5, 100)

    # Initialize W and scaling factors
    # W = np.ones((n_factors, n_genes))
    # W = np.ones((n_factors, n_genes))
    # they do!
    #   W = matrix(colSums(x), ncol=ncol(x), nrow=ncol(f), byrow=TRUE)/sum(f); # Init W (components, col=genes)
    W = np.tile(x.sum(axis=0), (n_factors, 1)) / f.sum()
    print(f'W init shape -> {W.shape} as f{(n_factors, n_genes)}')

    #W_init = W.copy()
    sc = np.clip(np.mean(x, axis=1) / np.mean(f @ W, axis=1), 0.1, 10)
    display(sc.shape)

    for iter in range(max_iter):
        if verbose:
            print(f"Iteration {iter + 1}")

        # Step 1: Fit weights W for each gene (columns of x)
        W_new = np.zeros_like(W)
        new_sigma = np.zeros(n_genes)
        total_ll = 0

        # parallizeable
        for i in range(n_genes):
            ll, s, w = fit_gene_weights(x[:, i], f, sc, sigma[i], init=W[:, i])
            W_new[:, i] = w
            new_sigma[i] = s
            total_ll += ll

            if verbose and (i % 250) == 0:
                print(f'     finished gene {i}')

        sigma = np.clip(new_sigma, 0.5, 1e3 - 10)
        W = W_new
        if verbose:
            print(f"  W log-likelihood: {-total_ll:.2f}")

        # Step 2: Fit scaling factors for each spot
        def fit_scaling_factor(i):
            def objective(log_sc):
                sc_i = np.exp(log_sc)
                mu = sc_i * (f[i, :] @ W)
                return -np.sum(negbinom_loglik(x[i, :], mu, sigma))
            res = minimize(objective, np.log(sc[i]), method='BFGS')
            return np.exp(res.x[0])

        sc = np.array([fit_scaling_factor(i) for i in range(n_spots)])

    return {
        'W': W.T,  # Return shape: [genes × components]
        'sigma': sigma,
        'sc': sc,
    }


In [80]:
expression_in_annotations(X_in.toarray(), test_cat)

W init shape -> (9, 21847) as f(9, 21847)


(3448,)

Iteration 1
     finished gene 0


KeyboardInterrupt: 

In [7]:
df_classif = pl.read_csv('../data/predicted_classifications.csv')
cnt = ad.read_h5ad('../data/anndata/counts/cnt-1.h5ad')

In [21]:
def filter_components(df: pl.DataFrame):
    '''
    Match the normalization and selection of the categories done in the code
    '''
    keep = ['Fat tissue','in situ','Lactiferous duct','Lymphoid nodule','Necrosis',
    "Tumor", "Stroma", "Lymphocyte", 'Vessels']

    frac = [
        (pl.col(c) / pl.sum_horizontal(keep)).alias(c)
        for c in keep
    ]

    return df.select('tnbc_id', 'slide_id', *keep).with_columns(frac)

filter_components(df_classif).filter(pl.col('tnbc_id') == 1)

tnbc_id,slide_id,Fat tissue,in situ,Lactiferous duct,Lymphoid nodule,Necrosis,Tumor,Stroma,Lymphocyte,Vessels
i64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64
1,"""CN1_C1.2x12""",0.407427,7.3638e-8,0.000083,1.4110e-8,0.004187,0.039167,0.540337,0.008677,0.00012
1,"""CN1_C1.2x14""",0.233833,0.000052,0.014814,1.4053e-7,0.002992,0.008759,0.731413,0.007158,0.000979
1,"""CN1_C1.2x16""",0.368589,0.000018,0.003944,6.1816e-8,0.001816,0.008884,0.607394,0.005602,0.003753
1,"""CN1_C1.2x18""",0.42156,0.000004,0.00264,4.2418e-10,0.019537,0.02242,0.521657,0.010491,0.001691
1,"""CN1_C1.2x20""",0.150813,0.000127,0.001888,3.6367e-8,0.006637,0.032204,0.784807,0.022083,0.001441
…,…,…,…,…,…,…,…,…,…,…
1,"""CN1_D1.65x23""",0.107657,0.000018,0.001438,9.2268e-9,0.001061,0.125051,0.748988,0.014764,0.001022
1,"""CN1_D1.65x25""",0.16,0.000027,0.007482,7.2132e-9,0.001403,0.12839,0.674768,0.027115,0.000815
1,"""CN1_D1.65x27""",0.066301,0.000473,0.004481,1.1622e-8,0.005165,0.465634,0.432142,0.025396,0.000408


In [26]:
def sort_categories(df: pl.DataFrame, adata: AnnData) -> pl.DataFrame:
    '''
    Once `df` is filtered to one sample, sort the rows to match
    the counts matrix
    '''
    order = pl.DataFrame({
        'slide_id': adata.obs_names,
        'sort_order': list(range(len(cnt.obs_names)))
    })

    return (
        df.join(order, on='slide_id')
            .sort('sort_order')
            .select(df.columns)
    )

In [29]:
filter_components(df_classif).filter(pl.col('tnbc_id') == 1).pipe(sort_categories, cnt)

#.drop('tnbc_id', 'slide_id').to_numpy()

tnbc_id,slide_id,Fat tissue,in situ,Lactiferous duct,Lymphoid nodule,Necrosis,Tumor,Stroma,Lymphocyte,Vessels
i64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64
1,"""CN1_C1.2x12""",0.407427,7.3638e-8,0.000083,1.4110e-8,0.004187,0.039167,0.540337,0.008677,0.00012
1,"""CN1_C1.2x14""",0.233833,0.000052,0.014814,1.4053e-7,0.002992,0.008759,0.731413,0.007158,0.000979
1,"""CN1_C1.2x16""",0.368589,0.000018,0.003944,6.1816e-8,0.001816,0.008884,0.607394,0.005602,0.003753
1,"""CN1_C1.2x18""",0.42156,0.000004,0.00264,4.2418e-10,0.019537,0.02242,0.521657,0.010491,0.001691
1,"""CN1_C1.2x20""",0.150813,0.000127,0.001888,3.6367e-8,0.006637,0.032204,0.784807,0.022083,0.001441
…,…,…,…,…,…,…,…,…,…,…
1,"""CN1_D1.65x23""",0.107657,0.000018,0.001438,9.2268e-9,0.001061,0.125051,0.748988,0.014764,0.001022
1,"""CN1_D1.65x25""",0.16,0.000027,0.007482,7.2132e-9,0.001403,0.12839,0.674768,0.027115,0.000815
1,"""CN1_D1.65x27""",0.066301,0.000473,0.004481,1.1622e-8,0.005165,0.465634,0.432142,0.025396,0.000408


In [17]:
from scipy.sparse import csc_array

In [30]:
test = csc_array(cnt.X)
test_cat = (
    filter_components(df_classif)
        .filter(pl.col('tnbc_id') == 1)
        .pipe(sort_categories, cnt)
        .drop('tnbc_id', 'slide_id')
        .to_numpy()
)

X_in = test[:, test.sum(axis=0) > 5]

In [81]:
cnt.obs

,x,y,new_x,new_y,pixel_x,pixel_y,slide
CN1_C1.2x12,2,12,2.04,12.05,197.507095,179.361456,CN1_C1
CN1_C1.2x14,2,14,2.05,14.08,197.594595,209.898956,CN1_C1
CN1_C1.2x16,2,16,2.03,16.05,197.294595,239.486456,CN1_C1
CN1_C1.2x18,2,18,2.03,18.04,197.307095,269.548956,CN1_C1
CN1_C1.2x20,2,20,2.07,20.04,197.857095,299.623956,CN1_C1
...,...,...,...,...,...,...,...
CN1_D1.65x23,65,23,64.99,22.98,252.269791,453.989599,CN1_D1
CN1_D1.65x25,65,25,65.00,25.00,257.385781,424.111567,CN1_D1
CN1_D1.65x27,65,27,64.95,26.99,263.310669,394.756950,CN1_D1
CN1_D1.65x29,65,29,65.01,29.00,267.748175,364.911597,CN1_D1
